In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q openai tqdm

import json
import time
import re
from tqdm import tqdm
from openai import OpenAI

INPUT_PATH = "/content/drive/MyDrive/conversation_cn.json"
OUTPUT_PATH = "/content/drive/MyDrive/conversation_cn_translated_en_deepseek_v4flash.json"

DEEPSEEK_API_KEY = ""

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

MODEL_NAME = "deepseek-v4-flash"


def extract_json_array(raw):
    raw = raw.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    try:
        return json.loads(raw)
    except Exception:
        match = re.search(r"\[.*\]", raw, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise


def translate_conversation_turns(turns, max_retries=5):
    items = []

    for idx, turn in enumerate(turns):
        role = turn.get("role", "unknown")
        intent = turn.get("intent") or "N/A"
        content = turn.get("content", "")

        if isinstance(content, str) and content.strip():
            items.append({
                "idx": idx,
                "role": role,
                "intent": intent,
                "content": content.strip()
            })

    if not items:
        return turns

    prompt = f"""
Translate the Chinese content fields in this e-commerce conversation into natural English.

Rules:
- Preserve each turn's meaning exactly.
- Preserve intent meaning, especially Revise, Inquire, Reveal, Chitchat, Interpret.
- Preserve correction, rejection, change of preference, refinement, or contrastive meaning.
- Do not add new product details.
- Do not remove negations, numbers, brands, colors, prices, sizes, or preferences.
- Keep translations concise and conversational.
- Return ONLY valid JSON.
- Return a JSON array with objects containing idx and content_en.
- Do not include markdown or explanation.

Input JSON:
{json.dumps(items, ensure_ascii=False)}
""".strip()

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a careful Chinese-to-English translator for intent classification datasets. Return only valid JSON."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.0,
                max_tokens=5000,
            )

            raw = response.choices[0].message.content.strip()
            translated_items = extract_json_array(raw)

            idx_to_translation = {
                int(x["idx"]): str(x["content_en"]).strip().strip('"').strip("'").strip()
                for x in translated_items
                if "idx" in x and "content_en" in x
            }

            expected_idxs = {item["idx"] for item in items}
            returned_idxs = set(idx_to_translation.keys())

            if expected_idxs != returned_idxs:
                raise ValueError(
                    f"Missing translations. Expected {expected_idxs}, got {returned_idxs}"
                )

            new_turns = []
            for idx, turn in enumerate(turns):
                new_turn = dict(turn)
                if idx in idx_to_translation:
                    new_turn["content"] = idx_to_translation[idx]
                new_turns.append(new_turn)

            return new_turns

        except Exception as e:
            wait = 2 ** attempt
            print(f"Conversation translation error, retrying in {wait}s: {e}")
            time.sleep(wait)

    print("Failed conversation translation, keeping original conversation")
    return turns


with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

translated_data = []

for i, conv in enumerate(tqdm(data, desc="Translating conversations")):
    new_conv = dict(conv)

    new_conv["conversation"] = translate_conversation_turns(
        conv["conversation"]
    )

    translated_data.append(new_conv)

    if i % 20 == 0:
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump(translated_data, f, ensure_ascii=False, indent=2)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(translated_data, f, ensure_ascii=False, indent=2)

print("Done. Saved to:", OUTPUT_PATH)

Translating conversations: 100%|██████████| 904/904 [1:16:20<00:00,  5.07s/it]


Done. Saved to: /content/drive/MyDrive/conversation_cn_translated_en_deepseek_v4flash.json


In [ ]:
# additonal chck

In [ ]:
import json
import re
from collections import Counter

PATH = "/content/drive/MyDrive/conversation_cn_translated_en_deepseek_v4flash.json"

with open(PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print("Total conversations:", len(data))

total_turns = 0
user_turns = 0
label_counter = Counter()

missing_content = 0
missing_role = 0
missing_intent = 0

chinese_remaining = 0
sample_chinese = []

def has_chinese(text):
    return bool(re.search(r'[\u4e00-\u9fff]', text))

for conv in data:
    for turn in conv.get("conversation", []):

        total_turns += 1

        role = turn.get("role")
        content = turn.get("content")
        intent = turn.get("intent")

        if role == "user":
            user_turns += 1

        if intent:
            label_counter[intent] += 1

        if not role:
            missing_role += 1

        if content is None or str(content).strip() == "":
            missing_content += 1

        if role == "user" and not intent:
            missing_intent += 1

        if isinstance(content, str) and has_chinese(content):
            chinese_remaining += 1

            if len(sample_chinese) < 10:
                sample_chinese.append(content)

print("\nTotal turns:", total_turns)
print("User turns:", user_turns)

print("\nLabel distribution:")
for k, v in sorted(label_counter.items()):
    print(f"{k}: {v}")

print("\nMissing fields:")
print("Missing role:", missing_role)
print("Missing content:", missing_content)
print("Missing user intent:", missing_intent)

print("\nChinese remaining in content:", chinese_remaining)

if sample_chinese:
    print("\nSample remaining Chinese:")
    for s in sample_chinese:
        print("-", s[:120])

print("\n" + "=" * 60)
print("SAMPLE CONVERSATIONS")
print("=" * 60)

for i in range(2):
    print(f"\nConversation {i}")
    print("-" * 50)

    for turn in data[i]["conversation"]:
        role = turn.get("role")
        intent = turn.get("intent", "")
        content = turn.get("content", "")

        print(f"{role.upper()} | {intent}")
        print(content[:200])
        print()

DATASET OVERVIEW
Total conversations: 904

Total turns: 5254
User turns: 3056

Label distribution:
Chitchat: 275
Inquire: 1
Interpret: 1590
Reveal: 975
Revise: 215

Missing fields:
Missing role: 0
Missing content: 0
Missing user intent: 0

Chinese remaining in content: 0

SAMPLE CONVERSATIONS

Conversation 0
--------------------------------------------------
USER | Chitchat
Hello

SYSTEM | 
What do you need?

USER | Reveal
I want to buy a phone

SYSTEM | 
How about these?

USER | Interpret
I want a Huawei phone

SYSTEM | 
How about these?

USER | Interpret
Great, thanks


Conversation 1
--------------------------------------------------
USER | Chitchat
Hello

SYSTEM | 
What can I help you with?

USER | Reveal
I want a fountain pen.

SYSTEM | 
How about these?

USER | Interpret
Very satisfied, thank you.

